In [5]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 64 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 30.10 KiB | 3.76 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [6]:
import os
import sys

REPO_ROOT = "/content/unstructuredata_tfidf_repo"
os.chdir(REPO_ROOT)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Project directory:", os.getcwd())
print("Files:", os.listdir(REPO_ROOT))

Project directory: /content/unstructuredata_tfidf_repo
Files: ['task_to_do.txt', 'src', 'LICENSE', 'data', '.github', 'notebooks', 'requirements.txt', 'tests', '.gitignore', '.git', 'README.md']


In [7]:
from src.load_data import load_documents

titles, documents = load_documents()

print(f"{len(documents)} documents loaded.\n")

for title, document in zip(titles, documents):
    print(f"- {title}: {len(document.split())} words")

# Verify the assignment requirements
assert len(documents) == 10, "Expected 10 documents."
assert all(len(doc.split()) > 30 for doc in documents), \
    "Every document must contain more than 30 words."

10 documents loaded.

- AI in Healthcare: 85 words
- Climate Change: 83 words
- Basketball Strategy: 85 words
- Italian Pasta: 91 words
- Space Exploration: 86 words
- Cybersecurity: 88 words
- Music Theory: 87 words
- Gardening Vegetables: 89 words
- Economics Inflation: 89 words
- Public Health: 91 words


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(feature_names))

TF-IDF matrix shape: (10, 537)
Vocabulary size: 537


In [9]:
import pandas as pd

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=titles,
    columns=feature_names
)

print("First 3 documents and first 10 terms:")
display(tfidf_df.iloc[:3, :10])

First 3 documents and first 10 terms:


,ability,accelerate,access,accountability,accuracy,accurate,actions,activity,adaptation,add
AI in Healthcare,0.000000,0.0,0.0,0.1246,0.1246,0.000000,0.000000,0.0,0.000000,0.0
Climate Change,0.000000,0.0,0.0,0.0000,0.0000,0.124795,0.124795,0.0,0.124795,0.0
Basketball Strategy,0.130137,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.000000,0.0


In [10]:
for title, row in tfidf_df.iterrows():
    top_terms = row.sort_values(ascending=False).head(5)

    print(f"\n{title}")
    for term, score in top_terms.items():
        print(f"  {term}: {score:.4f}")


AI in Healthcare
  ai: 0.3738
  healthcare: 0.2492
  use: 0.1648
  waiting: 0.1246
  transforming: 0.1246

Climate Change
  change: 0.2496
  emissions: 0.2496
  energy: 0.2122
  climate: 0.2122
  burning: 0.1248

Basketball Strategy
  decide: 0.1301
  switch: 0.1301
  player: 0.1301
  situations: 0.1301
  pick: 0.1301

Italian Pasta
  pasta: 0.4431
  sauces: 0.3323
  sauce: 0.2215
  different: 0.1883
  tubes: 0.1108

Space Exploration
  space: 0.3490
  mars: 0.2327
  missions: 0.2327
  agencies: 0.1163
  advanced: 0.1163

Cybersecurity
  cybersecurity: 0.2494
  systems: 0.2120
  data: 0.1649
  defenders: 0.1247
  attackers: 0.1247

Music Theory
  musicians: 0.2470
  music: 0.2470
  practice: 0.2099
  mix: 0.1235
  memory: 0.1235

Gardening Vegetables
  soil: 0.3581
  year: 0.2387
  pests: 0.2387
  improves: 0.2029
  rotate: 0.1194

Economics Inflation
  spending: 0.2345
  inflation: 0.2345
  trade: 0.1994
  economic: 0.1994
  income: 0.1173

Public Health
  health: 0.3462
  public: 0.

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def search_documents(query):
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    results = pd.DataFrame({
        "Title": titles,
        "Similarity Score": scores
    })

    return results.sort_values(
        by="Similarity Score",
        ascending=False
    ).reset_index(drop=True)

In [12]:
queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet"
]

for query in queries:
    print(f"\nQuery: {query}")
    display(search_documents(query))


Query: renewable energy and climate policy


,Title,Similarity Score
0,Climate Change,0.323458
1,Economics Inflation,0.095225
2,Gardening Vegetables,0.048458
3,AI in Healthcare,0.000000
4,Basketball Strategy,0.000000
5,Italian Pasta,0.000000
6,Cybersecurity,0.000000
7,Space Exploration,0.000000
8,Music Theory,0.000000
9,Public Health,0.000000



Query: password security and phishing


,Title,Similarity Score
0,Cybersecurity,0.176375
1,AI in Healthcare,0.000000
2,Climate Change,0.000000
3,Basketball Strategy,0.000000
4,Italian Pasta,0.000000
5,Space Exploration,0.000000
6,Music Theory,0.000000
7,Gardening Vegetables,0.000000
8,Economics Inflation,0.000000
9,Public Health,0.000000



Query: healthy sleep exercise and diet


,Title,Similarity Score
0,Public Health,0.284861
1,Gardening Vegetables,0.044702
2,Climate Change,0.000000
3,AI in Healthcare,0.000000
4,Basketball Strategy,0.000000
5,Italian Pasta,0.000000
6,Cybersecurity,0.000000
7,Space Exploration,0.000000
8,Music Theory,0.000000
9,Economics Inflation,0.000000


In [13]:
expected_titles = [
    "Climate Change",
    "Cybersecurity",
    "Public Health"
]

for query, expected in zip(queries, expected_titles):
    results = search_documents(query)
    actual = results.iloc[0]["Title"]
    status = "PASS" if actual == expected else "CHECK"
    print(f"{status}: {query}")
    print(f"Expected: {expected} | Actual: {actual}\n")

PASS: renewable energy and climate policy
Expected: Climate Change | Actual: Climate Change

PASS: password security and phishing
Expected: Cybersecurity | Actual: Cybersecurity

PASS: healthy sleep exercise and diet
Expected: Public Health | Actual: Public Health

